# MCP Server Basics: Building a Minimal MCP Server

## Table of contents
- [What a minimal MCP server exposes](#what-a-minimal-mcp-server-exposes)
- [Prerequisites](#prerequisites)
- [Imports and server creation](#imports-and-server-creation)
- [Registering a Tool](#registering-a-tool)
- [Registering a Resource](#registering-a-resource)
- [Registering a Prompt](#registering-a-prompt)
- [Testing with InMemoryTransport](#testing-with-inmemorytransport)
- [Transport selection: stdio vs SSE](#transport-selection-stdio-vs-sse)
- [.mcp.json configuration](#mcpjson-configuration)
- [Quick Reference](#quick-reference)

## What a minimal MCP server exposes

An MCP server is a process that advertises capabilities to connected clients using a standardized protocol.
A server can expose any combination of the three primitives:

```
MCP Server
├── Tools     — callable functions with side effects (model-controlled)
├── Resources — read-only data URIs (app-controlled)
└── Prompts   — reusable workflow templates (user-controlled)
```

Clients discover capabilities via `list` requests, then act on them:

| Request | What it does |
|---|---|
| `tools/list` | Returns registered Tool schemas |
| `tools/call` | Executes a Tool by name + arguments |
| `resources/list` | Returns registered Resource URIs |
| `resources/read` | Returns content at a given URI |
| `prompts/list` | Returns registered Prompt definitions |
| `prompts/get` | Renders a Prompt template with supplied arguments |

> The `@modelcontextprotocol/sdk` TypeScript library provides `Server`, `Client`,
> and `InMemoryTransport` for self-contained testing without a subprocess.

## Prerequisites

This notebook requires the MCP TypeScript SDK.

```bash
# Option 1: pin version in deno.json (recommended)
deno add npm:@modelcontextprotocol/sdk

# Option 2: Deno auto-downloads on first import — just run the cells
```

Key exports used in this notebook:

| Import path | Exports |
|---|---|
| `@modelcontextprotocol/sdk/server/index.js` | `Server` |
| `@modelcontextprotocol/sdk/client/index.js` | `Client` |
| `@modelcontextprotocol/sdk/inMemory.js` | `InMemoryTransport` |
| `@modelcontextprotocol/sdk/types.js` | Request schemas (`ListToolsRequestSchema`, etc.) |

In [ ]:
import { Server } from 'npm:@modelcontextprotocol/sdk/server/index.js';
import { Client } from 'npm:@modelcontextprotocol/sdk/client/index.js';
import { InMemoryTransport } from 'npm:@modelcontextprotocol/sdk/inMemory.js';
import {
  ListToolsRequestSchema,
  CallToolRequestSchema,
  ListResourcesRequestSchema,
  ReadResourceRequestSchema,
  ListPromptsRequestSchema,
  GetPromptRequestSchema,
} from 'npm:@modelcontextprotocol/sdk/types.js';

// Declare capabilities upfront — the server won't advertise primitives
// that aren't listed here, even if handlers are registered.
const server = new Server(
  { name: 'portfolio-mcp-server', version: '1.0.0' },
  { capabilities: { tools: {}, resources: {}, prompts: {} } },
);

console.log('Server created with capabilities: tools, resources, prompts');

## Registering a Tool

Tools are registered with **two handlers**:

| Handler schema | MCP request | Returns |
|---|---|---|
| `ListToolsRequestSchema` | `tools/list` | Array of Tool definitions (name + description + inputSchema) |
| `CallToolRequestSchema` | `tools/call` | `{ content: ToolContent[], isError?: boolean }` |

**Exam cross-point**: The Tool `inputSchema` follows the same design rules as raw `tool_use` —
proper `description`, `required` array, typed `properties`. Broken-schema questions in D2
can appear in either raw `tool_use` or MCP Tool context.

In [ ]:
// tools/list — returns Tool schemas (what Claude sees when it discovers tools)
server.setRequestHandler(ListToolsRequestSchema, async () => ({
  tools: [
    {
      name: 'get_portfolio_return',
      description:
        'Get portfolio return percentage for a time period. ' +
        'Use this when the user asks about investment performance.',
      inputSchema: {
        type: 'object' as const,
        properties: {
          period: { type: 'string', description: 'Quarter, e.g. "Q1 2026" or full year "2025"' },
        },
        required: ['period'],
      },
    },
  ],
}));

// tools/call — executes the requested tool and returns content
server.setRequestHandler(CallToolRequestSchema, async (request) => {
  const { name, arguments: args } = request.params;
  if (name === 'get_portfolio_return') {
    const { period } = args as { period: string };
    const returns: Record<string, number> = { 'Q1 2026': 4.2, '2025': 18.5, 'Q4 2025': 5.1 };
    const value = returns[period] ?? 7.3;
    return {
      content: [{ type: 'text' as const, text: `Portfolio return for ${period}: ${value}%` }],
    };
  }
  return {
    content: [{ type: 'text' as const, text: `Unknown tool: ${name}` }],
    isError: true,
  };
});

console.log('Tool registered: get_portfolio_return');

## Registering a Resource

Resources are registered with **two handlers**:

| Handler schema | MCP request | Returns |
|---|---|---|
| `ListResourcesRequestSchema` | `resources/list` | Array of Resource metadata (uri, name, mimeType) |
| `ReadResourceRequestSchema` | `resources/read` | `{ contents: [{ uri, text?, blob?, mimeType }] }` |

**URI conventions used in exam questions**:
- `file:///path/to/file` — local files
- `db://table/id` — database records (e.g. `db://users/42`)
- `api://path` — API-backed live data (e.g. `api://config/current`)

**Content format**: Use `text` for text content; `blob` (base64) for binary content.

In [ ]:
// resources/list — returns Resource metadata (URIs the client can read)
server.setRequestHandler(ListResourcesRequestSchema, async () => ({
  resources: [
    {
      uri: 'file:///data/portfolio.csv',
      name: 'portfolio-snapshot',
      description: 'Current portfolio holdings as of last market close',
      mimeType: 'text/csv',
    },
  ],
}));

// resources/read — returns the actual content at the requested URI
server.setRequestHandler(ReadResourceRequestSchema, async (request) => {
  const { uri } = request.params;
  if (uri === 'file:///data/portfolio.csv') {
    return {
      contents: [
        {
          uri,
          text: 'ticker,shares,price\nAAPL,100,189.30\nGOOGL,50,175.80\nMSFT,75,415.20',
          mimeType: 'text/csv',
        },
      ],
    };
  }
  throw new Error(`Resource not found: ${uri}`);
});

console.log('Resource registered: file:///data/portfolio.csv');

## Registering a Prompt

Prompts are registered with **two handlers**:

| Handler schema | MCP request | Returns |
|---|---|---|
| `ListPromptsRequestSchema` | `prompts/list` | Array of Prompt definitions with argument schemas |
| `GetPromptRequestSchema` | `prompts/get` | `{ description?, messages: PromptMessage[] }` |

**Runtime flow**:
1. UI calls `prompts/list` → shows user a menu of available Prompts
2. User selects a Prompt (optionally fills in arguments)
3. Host calls `prompts/get` with name + arguments
4. Server returns rendered `messages` array
5. Host injects those messages into the conversation

**Key**: The *user* triggers this flow — not Claude. That's why it's user-controlled.

In [ ]:
// prompts/list — returns Prompt definitions (shown in the UI as a menu)
server.setRequestHandler(ListPromptsRequestSchema, async () => ({
  prompts: [
    {
      name: 'quarterly-review',
      description: 'Standard quarterly portfolio review workflow — user selects from UI',
      arguments: [
        { name: 'quarter', description: 'Quarter to analyze, e.g. "Q1 2026"', required: true },
        { name: 'focus', description: 'Analysis focus: "returns" | "risk" | "recommendations"', required: false },
      ],
    },
  ],
}));

// prompts/get — renders the template with the supplied arguments
server.setRequestHandler(GetPromptRequestSchema, async (request) => {
  const { name, arguments: args } = request.params;
  if (name === 'quarterly-review') {
    const quarter = (args?.quarter as string | undefined) ?? 'this quarter';
    const focusPart = args?.focus ? ` Focus area: ${args.focus as string}.` : '';
    return {
      description: 'Quarterly portfolio review',
      messages: [
        {
          role: 'user' as const,
          content: {
            type: 'text' as const,
            text:
              `Please analyze the portfolio performance for ${quarter}.${focusPart} ` +
              'Provide key metrics, trends, and actionable recommendations.',
          },
        },
      ],
    };
  }
  throw new Error(`Prompt not found: ${name}`);
});

console.log('Prompt registered: quarterly-review');

## Testing with InMemoryTransport

`InMemoryTransport` connects a Client to a Server **in the same process** —
no subprocess, no HTTP, no stdio pipes. It is the standard tool for notebook demos and unit tests.

```
InMemoryTransport.createLinkedPair()
  ├── serverTransport → server.connect(serverTransport)
  └── clientTransport → client.connect(clientTransport)
```

Messages flow in-memory: client requests arrive at the server handler; server responses
flow back through the linked pair. The full MCP protocol is exercised.

In [ ]:
// Wire up Client ↔ Server with InMemoryTransport (no subprocess needed)
const [clientTransport, serverTransport] = InMemoryTransport.createLinkedPair();

await server.connect(serverTransport);

const client = new Client(
  { name: 'test-client', version: '1.0.0' },
  { capabilities: {} },
);
await client.connect(clientTransport);

console.log('=== Client connected via InMemoryTransport ===\n');

// ── Tool ─────────────────────────────────────────────────────────────────────
const { tools } = await client.listTools();
console.log('tools/list  :', tools.map(t => t.name));

const toolResult = await client.callTool({
  name: 'get_portfolio_return',
  arguments: { period: 'Q1 2026' },
});
const toolText = (toolResult.content as Array<{ type: string; text?: string }>)[0]?.text ?? '';
console.log('tools/call  :', toolText);

// ── Resource ──────────────────────────────────────────────────────────────────
const { resources } = await client.listResources();
console.log('\nresources/list:', resources.map(r => r.uri));

const resourceResult = await client.readResource({ uri: 'file:///data/portfolio.csv' });
const csvText = (resourceResult.contents[0] as { uri: string; text?: string }).text ?? '';
console.log('resources/read (first row):', csvText.split('\n')[0]);

// ── Prompt ────────────────────────────────────────────────────────────────────
const { prompts } = await client.listPrompts();
console.log('\nprompts/list:', prompts.map(p => p.name));

const promptResult = await client.getPrompt({
  name: 'quarterly-review',
  arguments: { quarter: 'Q1 2026', focus: 'risk' },
});
const promptText = (promptResult.messages[0].content as { type: string; text?: string }).text ?? '';
console.log('prompts/get :', promptText.slice(0, 90) + '...');

## Transport selection: stdio vs SSE

| Transport | When to use | How it works |
|---|---|---|
| **stdio** | Local — server on the same machine | Client spawns server as subprocess, communicates via stdin/stdout |
| **SSE** (Streamable HTTP) | Remote — cloud deployment or multiple clients | Client opens HTTPS connection; server streams responses via Server-Sent Events |

### Decision rule

```
Server on same machine as client?        →  stdio
Server on a different machine / cloud?   →  SSE
Multiple clients connecting?             →  SSE
Production deployment?                   →  SSE
Local development / testing?             →  stdio
```

> **Exam trap**: "production" almost always implies SSE — cloud deployment is the trigger,
> not just the number of clients. A single-client production cloud server = SSE.

In [ ]:
// .mcp.json — configuration file that Claude Code reads to register MCP servers
// Place in project root (.mcp.json) or home directory (~/.mcp.json)

const mcpConfigStdio = {
  mcpServers: {
    'portfolio-server': {
      command: 'deno',
      args: ['run', '--allow-net', '--allow-read', 'portfolio_mcp_server.ts'],
      transport: 'stdio',   // local subprocess
    },
  },
};

const mcpConfigSse = {
  mcpServers: {
    'portfolio-server': {
      url: 'https://portfolio-api.example.com/mcp',
      transport: 'sse',     // remote HTTPS endpoint
    },
  },
};

console.log('stdio .mcp.json (local / same-machine):');
console.log(JSON.stringify(mcpConfigStdio, null, 2));
console.log('\nSSE .mcp.json (remote / cloud / multi-client):');
console.log(JSON.stringify(mcpConfigSse, null, 2));

// ── Transport scenario exercises ─────────────────────────────────────────────
type Transport = 'stdio' | 'SSE';
interface TransportScenario {
  scenario: string;
  answer: Transport;
  reason: string;
}

const scenarios: TransportScenario[] = [
  {
    scenario: 'MCP server runs on the same local machine as the AI client',
    answer: 'stdio',
    reason: 'Same machine = stdio (subprocess communication)',
  },
  {
    scenario: 'MCP server deployed to AWS, accessed by multiple teams',
    answer: 'SSE',
    reason: 'Remote + multi-client = SSE (Streamable HTTP)',
  },
  {
    scenario: 'Production deployment serving 50 concurrent AI clients',
    answer: 'SSE',
    reason: 'Cloud / production / multiple clients = SSE',
  },
  {
    scenario: 'Developer testing a new MCP server locally during development',
    answer: 'stdio',
    reason: 'Local / development = stdio (no network overhead)',
  },
  {
    scenario: 'Single-tenant cloud SaaS AI product with one enterprise customer',
    answer: 'SSE',
    reason: 'Cloud deployment = SSE, regardless of client count',
  },
];

console.log('\n=== Transport Selection Exercises ===\n');
for (const [i, s] of scenarios.entries()) {
  console.log(`Q${i + 1}: ${s.scenario}`);
  console.log(`  Answer : ${s.answer}`);
  console.log(`  Reason : ${s.reason}`);
  console.log();
}

## Quick Reference

### MCP server handler pairs

| Primitive | List handler | Action handler |
|---|---|---|
| Tool | `tools/list` → `Tool[]` | `tools/call` → `{ content[], isError? }` |
| Resource | `resources/list` → `Resource[]` | `resources/read` → `{ contents[] }` |
| Prompt | `prompts/list` → `Prompt[]` | `prompts/get` → `{ messages[] }` |

### Capabilities declaration (exam trap)

```typescript
// If a primitive is missing from capabilities, clients won't discover it
// even if the handler is registered:
new Server(info, { capabilities: { tools: {} } })
//                                ^^^^^^^^^^^ resources and prompts hidden!
```

### Transport rules

| Keyword in scenario | Transport |
|---|---|
| local, same machine, development, subprocess | **stdio** |
| remote, cloud, AWS/GCP/Azure, production, multi-client | **SSE** |

### .mcp.json config shapes

```json
{ "mcpServers": { "name": { "command": "...", "args": [...], "transport": "stdio" } } }
{ "mcpServers": { "name": { "url": "https://...", "transport": "sse" } } }
```

### Next notebooks

- `03_tool_schema_design.ipynb` — spot and fix broken tool schemas (high-frequency exam pattern)
- `04_mcp_client_with_claude.ipynb` — end-to-end: Claude + MCP client + error handling